# Gymnasium 소개와 CartPole 데모

**예상 소요 시간:** 30분

Gymnasium 라이브러리를 사용해 CartPole 환경을 처음 실행해봅니다.

## 목표
- Gymnasium 환경 생성 방법 이해
- CartPole-v1의 기본 구조 파악
- `reset()`과 `step()` API 사용법 익히기

## 1. Gymnasium 설치 및 환경 생성

In [ ]:
import gymnasium as gym

# CartPole 환경 생성: 막대를 세운 채로 유지하는 게임
env = gym.make("CartPole-v1")

print("환경 생성 완료:", env.spec.id)

# 환경을 초기 상태로 리셋
state, info = env.reset()

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from collections import deque
import random

class DQN(nn.Module):
    def __init__(self, state_dim=4, hidden=16, action_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )
    def forward(self, x):
        return self.net(x)

def dqn_train(episodes=600, lr=1e-2, gamma=0.99, epsilon_start=1.0, epsilon_end=0.05, buffer_size=1000, batch=32):
    model = DQN(state_dim=4, hidden=16, action_dim=2)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    buffer = deque(maxlen=buffer_size)
    eps_decay = (epsilon_start - epsilon_end) / episodes
    first_ep_frames, last_ep_frames = [], []
    
    for ep in range(episodes):
        record = (ep == 0 or ep == episodes - 1)
        env = gym.make("CartPole-v1", render_mode="rgb_array" if record else None)
        state, _ = env.reset(seed=ep)
        total_r = 0
        frames = []
        if record:
            frames.append(env.render())
        eps = max(epsilon_end, epsilon_start - eps_decay * ep)
        done = False
        
        while not done:
            s = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                q = model(s)
            action = env.action_space.sample() if random.random() < eps else q.argmax(1).item()
            next_state, reward, terminated, truncated, _ = env.step(action)
            if record:
                frames.append(env.render())
            done = terminated or truncated
            buffer.append((state, action, reward, next_state, done))
            total_r += reward
            
            if len(buffer) >= batch:
                batch_data = random.sample(buffer, batch)
                ss = torch.FloatTensor(np.array([b[0] for b in batch_data]))
                aa = torch.LongTensor([b[1] for b in batch_data])
                rr = torch.FloatTensor(np.array([b[2] for b in batch_data]))
                ns = torch.FloatTensor(np.array([b[3] for b in batch_data]))
                dd = torch.FloatTensor(np.array([b[4] for b in batch_data]))
                q_pred = model(ss).gather(1, aa.unsqueeze(1)).squeeze()
                with torch.no_grad():
                    q_next = model(ns).max(1)[0]
                    target = rr + gamma * (1 - dd) * q_next
                loss = nn.functional.mse_loss(q_pred, target)
                opt.zero_grad()
                loss.backward()
                opt.step()
            
            state = next_state
        
        if ep == 0:
            first_ep_frames = frames
        elif ep == episodes - 1:
            last_ep_frames = frames
        env.close()
        if (ep + 1) % 100 == 0:
            print(f"Episode {ep+1}, reward: {total_r}")
    
    return model, first_ep_frames, last_ep_frames

In [ ]:
model, first_frames, last_frames = dqn_train(episodes=2000)

In [ ]:
import imageio

imageio.mimsave("first_episode.gif", first_frames, fps=30, loop=0)
imageio.mimsave("last_episode.gif", last_frames, fps=30, loop=0)

In [ ]:
from IPython.display import Image, display
display(Image("first_episode.gif"))
display(Image("last_episode.gif"))